# Tutorial 11 — Diffusion Models, from a Circle to a Torus

**MM845 — Tópicos de Geometria III: AI for Geometry**
Paired with **Lecture 11: Geometry-Aware ML II — Manifolds & Diffusion Models**

---

Every model so far approximated a *function*: a label, a potential, a value. Lecture
11 changes the task. We are given samples $x_1,\dots,x_N \sim p_{\text{data}}$ and asked
for a machine that produces **new samples from the same measure**. There is no label;
the examples are the training data.

Diffusion models do this in four separate steps, and this tutorial takes them one at a
time, on points that lie on a circle:

| step | what happens | learned? |
|---|---|---|
| **corrupt** | a fixed Gaussian chain blurs $p_{\text{data}}$ into $\mathcal N(0,I)$ | no |
| **score** | at each noise level, $\nabla_x \log p_t$ points back towards data | — |
| **denoise** | a network trained to predict the added noise *is* an estimate of the score | yes |
| **sample** | many small learned steps turn fresh noise into new data | no (uses the network) |

The thread of the course is that we grade methods against answers we know. Here that is
unusually satisfying: for points on a circle **the noised density and its score have a
closed form**, in Bessel functions. So we can check the learned score directly,
separate the error of learning from the error of sampling, and see precisely where a
trained diffusion model is weak. The last section moves to the torus and diffuses
*on the manifold itself*.

| § | Question | Lecture 11 |
|---|---|---|
| 1 | What does noising do to a measure, and what is its score exactly? | slides 7–8 |
| 2 | Why does predicting noise learn the score — and how well? | slide 9 |
| 3 | Sampling backwards: separating learning error from sampling error | slide 10 |
| 4 | Diffusion *on* a manifold: the torus, its heat kernel, and its spectrum | slides 3, 11 |
| 5 | What to take away | |

You need the `aigeo` environment from [Tutorial 1](../tutorial_01/README.md).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from scipy.special import ive, i0e, logsumexp

SEED = 20261005
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

GEO_DARK, GEO_TEAL, GEO_RUST = "#103158", "#006c86", "#b2461e"
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.titlesize": 10,
                     "axes.grid": True, "grid.alpha": 0.25,
                     "axes.prop_cycle": plt.cycler(color=[GEO_DARK, GEO_TEAL, GEO_RUST])})
print("torch", torch.__version__)

---
## 1. Corrupt the data, and compute the score exactly

**The data.** Points *on* the unit circle $S^1 \subset \mathbb{R}^2$, whose angles
follow a mixture of two von Mises laws: $70\%$ of the mass in a broad bump around
$\theta = 0$, $30\%$ in a narrow bump around $\theta = \pi$. The measure is singular —
concentrated on a curve of measure zero — and it is *not* uniform along that curve.
Both features will matter.

**The forward chain** is fixed; nothing about it is learned. For noise levels
$t = 1,\dots,T$ (levels, not optimisation iterations) and small variances $\beta_t$,

$$x_t = \sqrt{1-\beta_t}\,x_{t-1} + \sqrt{\beta_t}\,\varepsilon_t,\qquad
\varepsilon_t \sim \mathcal N(0,I).$$

Composing Gaussians gives a shortcut to any level:
$x_t = \alpha_t x_0 + \sigma_t\varepsilon$, where $\alpha_t = \prod_{j\le t}\sqrt{1-\beta_j}$
is the cumulative **signal** coefficient and $\sigma_t^2 = 1-\alpha_t^2$. The schedule
is chosen so that $\alpha_T \approx 0$, i.e. $p_T \approx \mathcal N(0,I)$.

The cell checks the shortcut against actually iterating the chain.

In [ ]:
T = 200
beta = np.linspace(1e-4, 0.06, T)                 # beta[t-1] is beta_t
alpha = np.sqrt(np.cumprod(1 - beta))             # cumulative signal coefficient
sigma = np.sqrt(1 - alpha**2)

W_MIX = np.array([0.7, 0.3])                      # von Mises mixture on the circle
MU = np.array([0.0, np.pi])
KAP = np.array([4.0, 8.0])
MU_VEC = np.stack([np.cos(MU), np.sin(MU)], axis=1)


def sample_data(n, rng):
    comp = rng.choice(2, size=n, p=W_MIX)
    th = rng.vonmises(MU[comp], KAP[comp])
    return np.stack([np.cos(th), np.sin(th)], axis=1)


def angle_density(th):
    return sum(w * np.exp(k * (np.cos(th - m) - 1)) / (2 * np.pi * i0e(k))
               for w, m, k in zip(W_MIX, MU, KAP))


print(f"alpha_1 = {alpha[0]:.4f}, sigma_1 = {sigma[0]:.4f}      "
      f"alpha_T = {alpha[-1]:.4f}, sigma_T = {sigma[-1]:.4f}")

# the shortcut x_t = alpha_t x_0 + sigma_t eps versus t steps of the chain
x0 = sample_data(200_000, rng)
t_chk = 60
x_chain = x0.copy()
for t in range(1, t_chk + 1):
    x_chain = np.sqrt(1 - beta[t - 1]) * x_chain + np.sqrt(beta[t - 1]) * rng.normal(size=x0.shape)
x_jump = alpha[t_chk - 1] * x0 + sigma[t_chk - 1] * rng.normal(size=x0.shape)
for name, arr in [("iterated chain", x_chain), ("shortcut      ", x_jump)]:
    print(f"t = {t_chk}, {name}:  E[x] = {arr.mean(0).round(3)}   E|x|^2 = {(arr**2).sum(1).mean():.4f}")
print(f"          exact:  E|x|^2 = alpha^2 + 2 sigma^2 = {alpha[t_chk-1]**2 + 2*sigma[t_chk-1]**2:.4f}")

### The score has a closed form here

The **score** at level $t$ is $s(x,t) = \nabla_x \log p_t(x)$: the direction in which
the log-density of the *noisy* data increases fastest. Note that any unknown
normalising constant disappears under $\nabla\log$.

For our data $p_t$ is the circle measure convolved with a Gaussian, and the angular
integral can be done exactly. With $u(\varphi) = (\cos\varphi,\sin\varphi)$ and a von
Mises component of concentration $k$ and mean direction $\mu$,

$$\int_0^{2\pi} e^{\,w\cdot u(\varphi)}\,\frac{d\varphi}{2\pi} = I_0(\lvert w\rvert),
\qquad w = \frac{\alpha_t}{\sigma_t^2}\,x + k\,\mu ,$$

so each component contributes $I_0(\lvert w\rvert)/I_0(k)$ and, using $I_0' = I_1$,

$$s(x,t) \;=\; -\frac{x}{\sigma_t^2} \;+\; \frac{\alpha_t}{\sigma_t^2}\sum_m \gamma_m(x)\,
\frac{I_1(\lvert w_m\rvert)}{I_0(\lvert w_m\rvert)}\,\frac{w_m}{\lvert w_m\rvert},$$

with $\gamma_m(x)$ the posterior probability of component $m$. The Bessel functions
overflow quickly, so the code uses their exponentially scaled versions. We verify the
formula against finite differences of $\log p_t$ — which we also have in closed form.

In [ ]:
def _components(x, t):
    a, s2 = alpha[t - 1], sigma[t - 1]**2
    w = (a / s2) * x[:, None, :] + (KAP[:, None] * MU_VEC)[None]      # (n, m, 2)
    A = np.linalg.norm(w, axis=-1)
    log_I0A = np.log(ive(0, A)) + A
    log_I0k = np.log(ive(0, KAP)) + KAP
    return a, s2, w, A, np.log(W_MIX) + log_I0A - log_I0k


def log_pt(x, t):
    """Exact log-density of the noised data at level t."""
    a, s2, _, _, lg = _components(x, t)
    return -np.log(2 * np.pi * s2) - ((x**2).sum(1) + a * a) / (2 * s2) + logsumexp(lg, axis=1)


def score(x, t):
    """Exact score grad_x log p_t(x)."""
    a, s2, w, A, lg = _components(x, t)
    gamma = np.exp(lg - logsumexp(lg, axis=1, keepdims=True))
    ratio = ive(1, A) / ive(0, A)
    return -x / s2 + (a / s2) * (gamma[..., None] * ratio[..., None] * w / A[..., None]).sum(1)


xq = rng.normal(size=(8, 2))
for t in (1, 20, 200):
    h = 1e-5
    fd = np.stack([(log_pt(xq + [h, 0], t) - log_pt(xq - [h, 0], t)) / (2 * h),
                   (log_pt(xq + [0, h], t) - log_pt(xq - [0, h], t)) / (2 * h)], axis=1)
    print(f"t = {t:3d}:  max |score - finite difference| = {np.abs(score(xq, t) - fd).max():.1e}"
          f"     (largest |score| = {np.abs(score(xq, t)).max():8.1f})")

In [ ]:
grid = np.linspace(-2.2, 2.2, 221)
GX, GY = np.meshgrid(grid, grid)
G = np.stack([GX.ravel(), GY.ravel()], axis=1)

fig, axes = plt.subplots(1, 5, figsize=(12.4, 2.8))
for ax, t in zip(axes, [0, 8, 30, 80, 200]):
    pts = sample_data(700, rng)
    if t > 0:
        pts = alpha[t - 1] * pts + sigma[t - 1] * rng.normal(size=pts.shape)
        P = np.exp(log_pt(G, t)).reshape(GX.shape)
        ax.contourf(GX, GY, P, levels=12, cmap="Blues", alpha=0.75)
    ax.scatter(*pts.T, s=2, color=GEO_RUST, alpha=0.6)
    ax.set_aspect("equal"); ax.set_xlim(-2.2, 2.2); ax.set_ylim(-2.2, 2.2)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title("data $p_{0}$" if t == 0 else f"$t = {t}$:  $\\alpha_t = {alpha[t-1]:.2f}$")
fig.suptitle("the forward chain: exact density $p_t$ (blue) and samples (rust)", y=1.04)
plt.tight_layout(); plt.show()

### What the score is — and what it is not

A tempting picture is that the score just points to the nearest point of the circle.
Slide 8 warns that it is not generally a nearest-point projection, and with an exact
formula we can see why. Decompose the score on the circle of radius $\alpha_t$ into a
**radial** part and a **tangential** part:

- the radial part pulls towards the circle;
- the tangential part pushes **along** the circle, towards high density. As
  $\sigma_t \to 0$ it should become exactly $\frac{1}{\alpha_t}\frac{d}{d\theta}\log\rho(\theta)$,
  the log-derivative of the angular density $\rho$.

In [ ]:
th = np.linspace(0, 2 * np.pi, 721)[:-1]
dlog_rho = np.gradient(np.log(angle_density(th)), th)

fig, axes = plt.subplots(1, 2, figsize=(10.2, 3.8))

t = 12
S_ = score(G, t)
axes[0].contourf(GX, GY, np.exp(log_pt(G, t)).reshape(GX.shape), levels=14, cmap="Blues")
axes[0].streamplot(grid, grid, S_[:, 0].reshape(GX.shape), S_[:, 1].reshape(GX.shape),
                   color=GEO_RUST, density=1.4, linewidth=0.7, arrowsize=0.8)
axes[0].set_aspect("equal"); axes[0].set_xticks([]); axes[0].set_yticks([])
axes[0].set_xlim(-2.2, 2.2); axes[0].set_ylim(-2.2, 2.2)
axes[0].set_title(f"integral curves of the score at $t = {t}$")

for t, col in [(2, GEO_DARK), (12, GEO_TEAL), (40, GEO_RUST)]:
    a = alpha[t - 1]
    on_circle = a * np.stack([np.cos(th), np.sin(th)], axis=1)
    s_ = score(on_circle, t)
    tang = -s_[:, 0] * np.sin(th) + s_[:, 1] * np.cos(th)
    axes[1].plot(th, tang, color=col, lw=1.6, label=f"tangential score, $\\sigma_t = {sigma[t-1]:.2f}$")
axes[1].plot(th, dlog_rho, "k--", lw=1.2, label=r"$\frac{d}{d\theta}\log\rho(\theta)$")
axes[1].set_xlabel(r"$\theta$"); axes[1].set_title("the score also flows along the circle")
axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

for t in (2, 12, 40):
    a = alpha[t - 1]
    s_ = score(a * np.stack([np.cos(th), np.sin(th)], axis=1), t)
    tang = -s_[:, 0] * np.sin(th) + s_[:, 1] * np.cos(th)
    rad = s_[:, 0] * np.cos(th) + s_[:, 1] * np.sin(th)
    print(f"t = {t:2d}:  max |tangential - (1/alpha) dlog rho| = {np.abs(tang - dlog_rho / a).max():.3f}"
          f"   (scale of dlog rho: {np.abs(dlog_rho).max():.2f});   mean radial score on the circle {rad.mean():+.2f}")

At low noise the tangential score coincides with $\frac{d}{d\theta}\log\rho$; as the
noise grows it becomes a smoothed version of it. So the score is really two things at
once: *get onto the manifold*, and *move along it towards where the data is dense*.
A nearest-point projection would do only the first, and would generate points spread
uniformly along the circle.

Notice too the printed radial score **on** the circle of radius $\alpha_t$: it is not
zero. The density's ridge sits slightly *inside* that circle. Points just inside a
curved set are near more of it than points just outside — a curvature effect you will
quantify in Exercise 1(a).

> **Exercise 1 — the geometry in the score.**
> (a) For *uniform* angles ($k = 0$), show from the formula that the radial score on
> the circle $r = \alpha_t$ tends to $-1/(2\alpha_t)$ as $\sigma_t \to 0$, so the ridge of
> $p_t$ sits at $r^\ast \approx \alpha_t - \sigma_t^2/(2\alpha_t)$. Check numerically. What
> is the role of the curvature $1/\alpha_t$?
>
> (b) Replace the circle by an ellipse, where there is no closed form. For a finite
> dataset, the exact score of $p_t$ is still computable: it is the score of a mixture of
> Gaussians centred at $\alpha_t x_i$. Plot it and compare with the circle.
>
> (c) Slide 6 said no diffeomorphism of $\mathbb R^2$ can push $\mathcal N(0,I)$ exactly
> onto the circle. Why not? What does a normalising flow trained on this data produce
> instead?

---
## 2. Learning the score by denoising

In practice $p_t$ is unknown, so the score must be learned. Slide 9's trick turns this
into ordinary regression: draw $x_0$ from the data, a level $t$, and noise $\varepsilon$;
form $x_t = \alpha_t x_0 + \sigma_t\varepsilon$; and train a network to predict
$\varepsilon$ from $(x_t, t)$ with squared loss. The target is known because we made
the noise ourselves.

The minimiser of that loss is the conditional mean $\varepsilon^\ast(x_t,t) =
\mathbb E[\varepsilon \mid x_t]$, and Tweedie's formula identifies it with the score,
$\varepsilon^\ast = -\sigma_t\, s(x_t,t)$. So $s_\theta = -\varepsilon_\theta/\sigma_t$ is a
score estimate.

This gives us something the lecture could not show: **the loss the network can
possibly reach**. Conditional expectation is orthogonal projection in $L^2$ — Tutorial
3 §1 again — so Pythagoras splits the loss exactly:

$$\underbrace{\mathbb E\lVert\varepsilon - \varepsilon_\theta\rVert^2}_{\text{training loss}}
\;=\; \underbrace{\mathbb E\lVert\varepsilon - \varepsilon^\ast\rVert^2}_{\text{irreducible floor}}
\;+\; \underbrace{\mathbb E\lVert\varepsilon^\ast - \varepsilon_\theta\rVert^2}_{\text{what training can remove}} .$$

The loss never reaches zero, and we can compute the floor because we know the score.

The network takes $(x_t, t)$, embedding $t$ with sinusoidal features — the positional
encoding of Lecture 6 — and is trained with the canonical loop of Lecture 3.

In [ ]:
class EpsNet(nn.Module):
    def __init__(self, width=128, n_freq=8):
        super().__init__()
        self.register_buffer("freqs", 2.0**torch.arange(n_freq) * np.pi)
        self.net = nn.Sequential(nn.Linear(2 + 2 * n_freq, width), nn.SiLU(),
                                 nn.Linear(width, width), nn.SiLU(),
                                 nn.Linear(width, width), nn.SiLU(),
                                 nn.Linear(width, 2))

    def forward(self, x, t):
        u = (t.float() / T)[:, None] * self.freqs[None]
        return self.net(torch.cat([x, torch.sin(u), torch.cos(u)], dim=1))


AL = torch.tensor(alpha, dtype=torch.float32)
SG = torch.tensor(sigma, dtype=torch.float32)
X_TRAIN = torch.tensor(sample_data(5000, np.random.default_rng(10)), dtype=torch.float32)

torch.manual_seed(0)
eps_net = EpsNet()
opt = torch.optim.Adam(eps_net.parameters(), lr=2e-3)
STEPS = 6000
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, STEPS)
for step in range(STEPS):
    idx = torch.randint(0, len(X_TRAIN), (512,))
    x0b = X_TRAIN[idx]
    tb = torch.randint(1, T + 1, (512,))
    eb = torch.randn_like(x0b)
    xtb = AL[tb - 1, None] * x0b + SG[tb - 1, None] * eb
    loss = ((eps_net(xtb, tb) - eb)**2).sum(1).mean()
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
print(f"trained {STEPS} steps on {len(X_TRAIN)} points; last batch loss {loss.item():.4f}")


@torch.no_grad()
def eps_hat(x, t):
    return eps_net(torch.tensor(x, dtype=torch.float32),
                   torch.full((len(x),), t, dtype=torch.long)).numpy().astype(float)


def learned_score(x, t):
    return -eps_hat(x, t) / sigma[t - 1]

In [ ]:
levels = [1, 2, 3, 5, 8, 12, 20, 30, 50, 80, 120, 160, 200]
floor, model_loss, rel_err = [], [], []
g_rng = np.random.default_rng(5)
for t in levels:
    x0g = sample_data(20000, g_rng)
    e = g_rng.normal(size=x0g.shape)
    xt = alpha[t - 1] * x0g + sigma[t - 1] * e
    e_star = -sigma[t - 1] * score(xt, t)                 # the ideal prediction
    e_th = eps_hat(xt, t)
    floor.append(((e - e_star)**2).sum(1).mean())
    model_loss.append(((e - e_th)**2).sum(1).mean())
    rel_err.append(((e_th - e_star)**2).sum(1).mean() / (e_star**2).sum(1).mean())

print(f"{'t':>4s} {'sigma_t':>8s} {'floor':>8s} {'model':>8s} {'rel. error':>11s}")
for t, f_, m_, r_ in zip(levels, floor, model_loss, rel_err):
    print(f"{t:4d} {sigma[t-1]:8.3f} {f_:8.4f} {m_:8.4f} {r_:11.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.2))
axes[0].semilogx(levels, model_loss, "o-", color=GEO_RUST, label="trained network")
axes[0].semilogx(levels, floor, "s-", color=GEO_DARK, label="irreducible floor (exact)")
axes[0].axhline(1.0, color=GEO_TEAL, ls="--", lw=1.2, label=r"$\dim S^1 = 1$")
axes[0].set_xlabel("noise level $t$"); axes[0].set_ylabel("denoising loss")
axes[0].set_title("denoising loss against what is achievable"); axes[0].legend(fontsize=7.5)
axes[1].loglog(levels, rel_err, "o-", color=GEO_RUST)
axes[1].set_xlabel("noise level $t$")
axes[1].set_ylabel(r"$\|\varepsilon_\theta - \varepsilon^\ast\|^2 / \|\varepsilon^\ast\|^2$")
axes[1].set_title("relative error of the learned score")
plt.tight_layout(); plt.show()

Three things in that table repay attention.

**The floor at low noise is the dimension of the manifold.** At $t = 1$ the irreducible
loss is $\approx 1$, and $1 = \dim S^1$. When $\sigma_t$ is small, the component of
$\varepsilon$ *normal* to the circle is recoverable — it is simply how far $x_t$ sits
from the circle, divided by $\sigma_t$. The *tangential* component is not: sliding
$x_0$ a little along the circle produces the same $x_t$, so no model can tell. The
normal directions are learnable, the tangent directions are not, and the floor counts
the tangent directions. A trained diffusion model therefore carries an estimate of
**intrinsic dimension** — Tutorial 2 §7's quantity, arriving from a completely
different direction. At high noise the floor falls to $0$, because $x_t$ is almost
pure noise and determines $\varepsilon$.

**Away from low noise the network is excellent.** From $t \approx 30$ upward its loss
sits on the floor and the relative error in the score is below half a percent; even at
$t = 5$ it is only about $3\%$.

**At the lowest levels it is poor.** There the ideal $\varepsilon^\ast$ changes by
order one over a distance $\sigma_1 = 0.01$ across the circle: it is by far the sharpest
function in the problem. A smooth MLP fits sharp functions last — this is the spectral
bias of Tutorial 4 §4 — and the error is then amplified by $1/\sigma_t$ in the score.
§3 shows the consequence.

> **Exercise 2 — the floor, and the hard end.**
> (a) Put the data on $S^2 \subset \mathbb{R}^3$, or on a circle embedded in
> $\mathbb{R}^3$, and check the floor at small $t$ tends to $2$ (respectively $1$). You
> have built an intrinsic-dimension estimator. What goes wrong if the data carry a
> little off-manifold noise of their own?
>
> (b) Train the same network to predict $x_0$ instead of $\varepsilon$ (and convert to a
> score). Compare the error at low and high noise, and explain which parametrisation
> suits which end.
>
> (c) Add Fourier features of $x$ to the input — the fix of Tutorial 4 Exercise 3(b) — and
> measure how much of the $t = 1$ error it removes.

---
## 3. Sampling backwards

With a score in hand, generation runs the chain backwards (slide 10). Draw
$x_T \sim \mathcal N(0,I)$ and for $t = T,\dots,1$ set

$$x_{t-1} \;=\; \frac{x_t + \beta_t\, s(x_t,t)}{\sqrt{1-\beta_t}} \;+\; \sqrt{\tilde\beta_t}\,z_t,
\qquad \tilde\beta_t = \beta_t\,\frac{\sigma_{t-1}^2}{\sigma_t^2},\quad \sigma_0 = 0 .$$

This Gaussian reverse step is an **approximation**, and it has two separate sources of
error: the score may be wrong, and finitely many Gaussian steps only approximate the
true reverse process. Usually the two cannot be told apart. Here they can — run the
sampler once with the **exact** score, and once with the **learned** one.

In [ ]:
def reverse_sample(score_fn, n, rng, x_start=None, t_start=T):
    x = rng.normal(size=(n, 2)) if x_start is None else np.repeat(x_start[None], n, axis=0)
    for t in range(t_start, 0, -1):
        b = beta[t - 1]
        bt = b * (sigma[t - 2]**2 if t > 1 else 0.0) / sigma[t - 1]**2
        x = (x + b * score_fn(x, t)) / np.sqrt(1 - b) + np.sqrt(bt) * rng.normal(size=x.shape)
    return x


BINS = np.linspace(0, 2 * np.pi, 73)
MID = 0.5 * (BINS[1:] + BINS[:-1])


def sample_quality(x):
    r = np.linalg.norm(x, axis=1)
    ang = np.arctan2(x[:, 1], x[:, 0]) % (2 * np.pi)
    h, _ = np.histogram(ang, bins=BINS, density=True)
    tv = 0.5 * np.sum(np.abs(h - angle_density(MID))) * (BINS[1] - BINS[0])
    return np.abs(r - 1).mean(), tv, np.mean(np.cos(ang) > 0)


N_GEN = 5000
x_exact = reverse_sample(score, N_GEN, np.random.default_rng(1))
x_learn = reverse_sample(learned_score, N_GEN, np.random.default_rng(1))
x_data = sample_data(N_GEN, np.random.default_rng(2))
p_heavy = np.mean(np.cos(np.arctan2(*sample_data(400_000, np.random.default_rng(3))[:, ::-1].T)) > 0)

print(f"{'':22s} {'mean |r-1|':>11s} {'angle TV':>9s} {'mass near theta=0':>18s}")
for name, xx in [("fresh data", x_data), ("exact score", x_exact), ("learned score", x_learn)]:
    e, tv, h = sample_quality(xx)
    print(f"{name:22s} {e:11.5f} {tv:9.4f} {h:18.3f}")
print(f"{'(true mass near 0)':22s} {'':11s} {'':9s} {p_heavy:18.3f}")

ang_l = np.arctan2(x_learn[:, 1], x_learn[:, 0]) % (2 * np.pi)
sparse = angle_density(ang_l) < 0.1
r_err = np.abs(np.linalg.norm(x_learn, axis=1) - 1)
print(f"\nlearned score, where the data are sparse (rho < 0.1): mean |r-1| = {r_err[sparse].mean():.4f}"
      f"   ({sparse.mean():.1%} of samples)")
print(f"learned score, where the data are dense  (rho >= 0.1): mean |r-1| = {r_err[~sparse].mean():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11.4, 3.4))
for ax, xx, name, col in [(axes[0], x_exact, "exact score", GEO_TEAL),
                          (axes[1], x_learn, "learned score", GEO_RUST)]:
    ax.scatter(*xx[:2500].T, s=2, alpha=0.5, color=col)
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="k", lw=0.6, ls="--"))
    ax.set_aspect("equal"); ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
    ax.set_title(f"generated, {name}")

for xx, name, col in [(x_exact, "exact score", GEO_TEAL), (x_learn, "learned score", GEO_RUST)]:
    ang = np.arctan2(xx[:, 1], xx[:, 0]) % (2 * np.pi)
    axes[2].hist(ang, bins=BINS, density=True, histtype="step", lw=1.5, color=col, label=name)
axes[2].plot(MID, angle_density(MID), "k--", lw=1.2, label=r"true $\rho(\theta)$")
axes[2].set_xlabel(r"$\theta$"); axes[2].set_title("angular distribution"); axes[2].legend(fontsize=7.5)
plt.tight_layout(); plt.show()

Read the table row by row.

- **Exact score.** The angular distance is at the level of *fresh data* — the floor set
  purely by using a finite sample — and the points lie on the circle to within
  $10^{-4}$. With the true score, $T = 200$ Gaussian steps lose essentially nothing.
- **Learned score.** The *angles* are nearly as good: both bumps, in the right
  proportion. But the points sit **near** the circle rather than on it. That residual is
  the low-noise score error of §2, and it lives in the last few steps, whose whole job
  is the final snap onto the manifold. It is also about twice as large on the sparse
  arcs between the bumps, where the network saw few training points — the score is
  learned only as well as the data covers it.

So the one weakness of the trained model is geometric and precisely located. It is
also the motivation for §4: when the manifold is *known*, we can stop asking a network
to learn the way back onto it.

### Generation is not un-noising

It is easy to read the reverse chain as undoing the noise that was added to a
particular training point. It is not (slide 10). Take one data point, noise it, and run
the reverse chain **many times from that single noisy point**.

In [ ]:
x_orig = np.array([1.0, 0.0])                       # a data point at theta = 0
fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.6))
for ax, t_start in zip(axes, [200, 25]):
    x_noisy = alpha[t_start - 1] * x_orig + sigma[t_start - 1] * np.random.default_rng(4).normal(size=2)
    ends = reverse_sample(score, 400, np.random.default_rng(6), x_start=x_noisy, t_start=t_start)
    back = np.mean(np.cos(np.arctan2(ends[:, 1], ends[:, 0])) > 0)
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="k", lw=0.6, ls="--"))
    ax.scatter(*ends.T, s=6, alpha=0.5, color=GEO_TEAL, label="400 reverse runs")
    ax.scatter(*x_orig, s=80, marker="*", color=GEO_RUST, zorder=5, label="original $x_0$")
    ax.scatter(*x_noisy, s=50, marker="x", color=GEO_DARK, zorder=5, label=f"noisy $x_{{{t_start}}}$")
    ax.set_aspect("equal"); ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.6, 1.6)
    ax.set_title(f"start at $t = {t_start}$:  {back:.0%} return to $x_0$'s side")
    ax.legend(fontsize=7, loc="lower left")
    print(f"from t = {t_start:3d} (alpha = {alpha[t_start-1]:.3f}): fraction on x_0's side {back:.3f}"
          f"   (base rate of that side under p_data: {p_heavy:.3f})")
plt.tight_layout(); plt.show()

From $t = 200$ the endpoints are spread over **both** bumps, in about the proportion the
data has — the chain has forgotten which point it started from. From $t = 25$ they
stay close to where $x_0$ was. Both are the same statement: run backwards from a noisy
$x_t$, the chain samples (approximately) from $p(x_0 \mid x_t)$, which is almost the
whole data distribution when little signal survives and concentrates as more does.
**Diffusion reverses distributions, not individual trajectories.**

> **Exercise 3 — the sampler.**
> (a) Implement the deterministic **probability-flow** sampler, which uses half the
> score coefficient and no fresh noise (Lecture 11's delivery notes). With the exact
> score it has the same marginals. Compare its angle TV and radial error with the
> stochastic sampler, for the exact and for the learned score.
>
> (b) Use fewer reverse steps by skipping levels, and plot angle TV against the number
> of steps for both scores. Where does discretisation error start to dominate?
>
> (c) Stop the learned sampler early, at $t = k$, and return the denoised estimate
> $\hat x_0 = (x_k + \sigma_k^2 s_\theta)/\alpha_k$. How does the radial error depend on
> $k$? Use §2's error curve to predict the best $k$ before measuring it.

---
## 4. Diffusion *on* the manifold: the torus

§3's learned samples missed the circle because they diffused in the ambient plane and
had to find their way back. Slide 11's alternative: when the manifold is known, **run
the whole process on it**. We take the flat torus $T^2 = \mathbb{R}^2 / 2\pi\mathbb{Z}^2$,
with angle coordinates $(\theta,\varphi)$, and data made of five bumps strung along the
curve $\varphi = 2\theta$ — a necklace that winds once one way and twice the other.

Three things change, and each is a piece of geometry.

**The noise is Brownian motion on $T^2$.** Its transition density is the **heat
kernel**, which on the flat torus is a *wrapped* Gaussian — sum over the images
$\theta + 2\pi k$. There is no shrinking towards the origin: the chain runs to its
stationary law, which is **uniform** Riemannian volume. Conveniently, a wrapped Gaussian
bump stays a wrapped Gaussian bump under heat flow, with variances adding, so the score
is again exact.

**The heat kernel is diagonal in the Laplacian's eigenbasis.** The eigenfunctions of
the flat Laplacian are $e^{i(k_1\theta + k_2\varphi)}$ with eigenvalue $\lvert k\rvert^2$,
and heat flow damps each by $e^{-\lvert k\rvert^2\sigma^2/2}$ — slide 3's spectral
representation, Tutorial 3's harmonics on $S^2$, now on $T^2$. The cell checks it.

**The representation splits in two.** The network *reads* the point through
$(\cos\theta,\sin\theta,\cos\varphi,\sin\varphi)$ — extrinsic, periodic features, so
whatever it outputs is automatically a well-defined function on $T^2$. The *dynamics*
use intrinsic angles, and a step is taken along the manifold: on the flat torus the
exponential map is addition modulo $2\pi$.

In [ ]:
TWO_PI = 2 * np.pi
wrap = lambda a: np.mod(a, TWO_PI)
wdiff = lambda a, b: np.mod(a - b + np.pi, TWO_PI) - np.pi    # signed, in (-pi, pi]

M_BUMPS, S_BUMP = 5, 0.22
CEN = np.stack([TWO_PI * np.arange(M_BUMPS) / M_BUMPS,
                wrap(2 * TWO_PI * np.arange(M_BUMPS) / M_BUMPS)], axis=1)
IMAGES = np.arange(-6, 7)


def sample_torus(n, rng):
    c = rng.integers(0, M_BUMPS, n)
    return wrap(CEN[c] + S_BUMP * rng.normal(size=(n, 2))), c


def wrapped_normal(delta, var):
    """log of the wrapped Gaussian (heat kernel on S^1) and its derivative."""
    z = delta[..., None] + TWO_PI * IMAGES
    v = np.asarray(var, dtype=float)[..., None]
    e = -z**2 / (2 * v)
    lse = logsumexp(e, axis=-1)
    w = np.exp(e - lse[..., None])
    return lse - 0.5 * np.log(TWO_PI * np.asarray(var, dtype=float)), (w * (-z / v)).sum(-1)


def torus_score(th, sig):
    """Exact score on T^2 of the data after heat flow for 'time' sigma^2."""
    lp, dl = wrapped_normal(wdiff(th[:, None, :], CEN[None]), S_BUMP**2 + sig**2)
    comp = lp.sum(-1)
    g = np.exp(comp - logsumexp(comp, axis=1, keepdims=True))
    return (g[..., None] * dl).sum(1)


# spectral check: heat flow multiplies Fourier mode k by exp(-|k|^2 sigma^2 / 2)
th0, _ = sample_torus(400_000, rng)
print(f"{'sigma':>6s} {'mode k':>8s} {'|k|^2':>6s} {'measured ratio':>15s} {'exp(-|k|^2 s^2/2)':>18s}")
for sig in (0.3, 0.6):
    tht = wrap(th0 + sig * rng.normal(size=th0.shape))
    for k in [(2, -1), (3, 1), (5, 0)]:
        c0 = np.abs(np.mean(np.exp(1j * (k[0] * th0[:, 0] + k[1] * th0[:, 1]))))
        ct = np.abs(np.mean(np.exp(1j * (k[0] * tht[:, 0] + k[1] * tht[:, 1]))))
        lam = k[0]**2 + k[1]**2
        print(f"{sig:6.1f} {str(k):>8s} {lam:6d} {ct / c0:15.4f} {np.exp(-lam * sig**2 / 2):18.4f}")

The measured ratios match $e^{-\lvert k\rvert^2\sigma^2/2}$: high-frequency structure is
erased first, at a rate set by the Laplacian's eigenvalue. (The modes shown are ones the
necklace actually contains — its five-fold symmetry kills the others.)

Now the learned version. Levels are indexed by $\sigma$ rather than $t$, and the network
predicts $\sigma\cdot$score, trained against the exact score of the wrapped heat kernel
started at each data point — denoising score matching, as in §2, but on $T^2$.

In [ ]:
N_LEVELS = 120
SIGS = np.geomspace(0.02, 3.5, N_LEVELS)             # increasing noise


class TorusScore(nn.Module):
    def __init__(self, width=128, n_freq=4):
        super().__init__()
        self.register_buffer("freqs", 2.0**torch.arange(n_freq) * np.pi)
        self.net = nn.Sequential(nn.Linear(4 + 2 * n_freq, width), nn.SiLU(),
                                 nn.Linear(width, width), nn.SiLU(),
                                 nn.Linear(width, width), nn.SiLU(),
                                 nn.Linear(width, 2))

    def forward(self, th, level):
        feats = torch.cat([torch.cos(th), torch.sin(th)], dim=1)     # periodic: lives on T^2
        u = (level.float() / N_LEVELS)[:, None] * self.freqs[None]
        return self.net(torch.cat([feats, torch.sin(u), torch.cos(u)], dim=1))


TH_TRAIN, _ = sample_torus(5000, np.random.default_rng(10))
torch.manual_seed(0)
tor_net = TorusScore()
opt = torch.optim.Adam(tor_net.parameters(), lr=2e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, STEPS)
b_rng = np.random.default_rng(11)
for step in range(STEPS):
    idx = b_rng.integers(0, len(TH_TRAIN), 512)
    lvl = b_rng.integers(0, N_LEVELS, 512)
    sig = SIGS[lvl]
    th_t = wrap(TH_TRAIN[idx] + sig[:, None] * b_rng.normal(size=(512, 2)))
    target = sig[:, None] * wrapped_normal(wdiff(th_t, TH_TRAIN[idx]), (sig**2)[:, None])[1]
    pred = tor_net(torch.tensor(th_t, dtype=torch.float32), torch.tensor(lvl))
    loss = ((pred - torch.tensor(target, dtype=torch.float32))**2).sum(1).mean()
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
print(f"trained {STEPS} steps; last batch loss {loss.item():.4f}")


@torch.no_grad()
def learned_torus_score(th, i):
    return tor_net(torch.tensor(th, dtype=torch.float32),
                   torch.full((len(th),), i, dtype=torch.long)).numpy() / SIGS[i]


def reverse_torus(score_fn, n, rng):
    th = rng.uniform(0, TWO_PI, (n, 2))                       # exact stationary law
    for i in range(N_LEVELS - 1, -1, -1):
        dv = SIGS[i]**2 - (SIGS[i - 1]**2 if i > 0 else 0.0)
        th = wrap(th + dv * score_fn(th, i) + np.sqrt(dv) * rng.normal(size=th.shape))
    return th


def torus_quality(th):
    d = np.linalg.norm(wdiff(th[:, None, :], CEN[None]), axis=-1)
    a = d.argmin(1)
    return np.bincount(a, minlength=M_BUMPS) / len(th), np.sqrt((wdiff(th, CEN[a])**2).mean())


th_exact = reverse_torus(lambda th, i: torus_score(th, SIGS[i]), N_GEN, np.random.default_rng(1))
th_learn = reverse_torus(learned_torus_score, N_GEN, np.random.default_rng(1))
th_data, lab_data = sample_torus(N_GEN, np.random.default_rng(2))
print(f"\n{'':15s} {'fraction in each bump':>36s} {'rms spread':>11s}")
for name, tt in [("fresh data", th_data), ("exact score", th_exact), ("learned score", th_learn)]:
    frac, spread = torus_quality(tt)
    print(f"{name:15s} {str(np.round(frac, 3)):>36s} {spread:11.4f}")
print(f"{'(true)':15s} {str([0.2] * M_BUMPS):>36s} {S_BUMP:11.4f}")

In [ ]:
def torus_embed(th, R=2.0, r=0.8):
    t, p = th[:, 0], th[:, 1]
    return np.stack([(R + r * np.cos(p)) * np.cos(t), (R + r * np.cos(p)) * np.sin(t), r * np.sin(p)], 1)


lab_learn = np.linalg.norm(wdiff(th_learn[:, None, :], CEN[None]), axis=-1).argmin(1)
cmap = plt.get_cmap("viridis", M_BUMPS)
curve = np.linspace(0, TWO_PI, 400)
curve_th = np.stack([curve, wrap(2 * curve)], axis=1)          # the (1,2) curve phi = 2 theta

fig = plt.figure(figsize=(12.0, 3.9))
for k, (tt, lab, name) in enumerate([(th_data, lab_data, "data"),
                                     (th_learn, lab_learn, "generated (learned score)")]):
    ax = fig.add_subplot(1, 3, k + 1)
    jumps = np.where(np.abs(np.diff(curve_th[:, 1])) > np.pi)[0] + 1
    for seg in np.split(curve_th, jumps):
        ax.plot(seg[:, 0], seg[:, 1], color="0.55", lw=0.8, ls="--")
    ax.scatter(tt[:, 0], tt[:, 1], c=lab, cmap=cmap, s=2, alpha=0.6)
    ax.set_xlim(0, TWO_PI); ax.set_ylim(0, TWO_PI); ax.set_aspect("equal")
    ax.set_xlabel(r"$\theta$"); ax.set_ylabel(r"$\varphi$"); ax.set_title(f"{name}: the flat square")

ax = fig.add_subplot(1, 3, 3, projection="3d")
U, V = np.meshgrid(np.linspace(0, TWO_PI, 48), np.linspace(0, TWO_PI, 24))
surf = torus_embed(np.stack([U.ravel(), V.ravel()], 1)).reshape(*U.shape, 3)
ax.plot_wireframe(surf[..., 0], surf[..., 1], surf[..., 2], color="0.7", lw=0.3, alpha=0.5)
C = torus_embed(curve_th)
ax.plot(*C.T, color="0.35", lw=1.0, ls="--")
E = torus_embed(th_learn[:3000])
ax.scatter(*E.T, c=lab_learn[:3000], cmap=cmap, s=3, alpha=0.8, depthshade=False)
ax.set_xlim(-2.9, 2.9); ax.set_ylim(-2.9, 2.9); ax.set_zlim(-1.3, 1.3)
ax.set_box_aspect((1, 1, 0.45)); ax.view_init(elev=32, azim=-60)
ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([]); ax.grid(False)
ax.set_title("the same samples on the embedded torus\n(dashed: the curve $\\varphi = 2\\theta$)")
plt.tight_layout(); plt.show()

The learned sampler places mass in all five bumps in the right proportions, with a
spread about ten per cent wider than the true one. And every one of its samples lies **exactly** on $T^2$,
not approximately — not because the network learned the manifold, but because every
step of the process was taken on it. Compare §3, where the same kind of network had to
learn the way back onto the circle and did not quite manage.

That is the practical content of slide 11: **when the geometry is known, build it into
the process, not the loss.** The network is left to learn only what is genuinely
unknown — where along the manifold the data lives.

> **Exercise 4 — generating geometry.**
> (a) **Conditional generation** (slide 11). Give the network a one-hot label $c$ for the
> bump each training point came from, train $s_\theta(\theta, \sigma, c)$, and generate
> from bump $3$ only. Verify the output independently, as the lecture insists: check the
> fractions rather than trusting the label.
>
> (b) **Diffusion on $S^2$.** Brownian motion on the sphere can be simulated with small
> tangent Gaussian steps followed by the exponential map. Check that it converges to the
> uniform measure using Tutorial 2's test — the height coordinate becomes uniform on
> $[-1,1]$ — and compare the rate with the first nonzero eigenvalue of $\Delta_{S^2}$.
>
> (c) **The extrinsic alternative.** Train §2's ambient model on the torus embedded in
> $\mathbb{R}^4$ via $(\cos\theta,\sin\theta,\cos\varphi,\sin\varphi)$ and measure how far
> its samples land from the torus. Which bumps suffer most, and why?

---
## 5. What to take away

- **Generative modelling learns a measure, not a function.** Diffusion does it in four
  separable steps — corrupt, score, denoise, sample — and only the third involves
  learning.
- **The score is geometry.** Near the data it pulls onto the manifold *and* flows along
  it towards high density. It is not a nearest-point projection, and its zero set is not
  even the manifold: curvature shifts the ridge inward.
- **Predicting noise learns the score**, and because conditional expectation is an
  orthogonal projection, the achievable loss is computable. At low noise that floor is
  the manifold's intrinsic dimension: normal noise is recoverable, tangent noise is not.
- **A trained model is weakest at low noise**, where the score is sharpest. With an exact
  score to compare against, that weakness shows up in exactly one place: samples near
  the manifold instead of on it.
- **Generation reverses distributions, not trajectories.** From a noisy point the chain
  samples $p(x_0\mid x_t)$, which forgets its starting point as the noise grows.
- **When the manifold is known, diffuse on it.** On the torus the heat kernel replaces
  the Gaussian, uniform volume replaces $\mathcal N(0,I)$, the exponential map replaces
  vector addition — and samples lie on the manifold by construction.

### Next

**Lecture 12** returns to approximating functions, but now the training signal is a
differential equation rather than data — **physics-informed neural networks**, of which
the neural Calabi–Yau metrics of slide 4 are an example. **Tutorial 12** solves an
elliptic PDE with one.

### Further reading

- Ho, Jain & Abbeel, "Denoising diffusion probabilistic models", *NeurIPS* 2020 — §2 and §3.
- Song et al., "Score-based generative modeling through stochastic differential equations", *ICLR* 2021 — the continuous-time view and the probability-flow ODE of Exercise 3(a).
- Efron, "Tweedie's formula and selection bias", *JASA* **106** (2011) — why predicting noise gives the score.
- De Bortoli et al., "Riemannian score-based generative modelling", *NeurIPS* 2022 — §4 on general manifolds.
- Stanczuk et al., "Your diffusion model secretly knows the dimension of the data manifold", arXiv:2212.12611 — the floor of §2, made into an estimator.